# M³ fast: all PANGAEA and iNaturalist datasets

This notebook runs the same **two-agent, one-revision-round** M³ path and SURF `default-text-large` model as `m3_thorough_20_comparison.ipynb`, on every CSV in the PANGAEA and iNaturalist experiment directories.

Results are written to `outputs/m3_fast_all_Qwen3.6-35B-A3B-FP8`. The 20-dataset folder `outputs/m3_fast_20_default-text-large` is read only: a file that already succeeded there is copied into this folder and is not run again. Rerunning this notebook skips successes already stored here. Failures stay retryable. Set `M3_MAX_NEW_RUNS=0` to refresh the manifest and reuse existing successes without new model calls.

In [1]:
import hashlib
import json
import os
import shutil
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from tqdm.auto import tqdm

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src' / 'config.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the metadata_agent checkout')
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / '.env')
os.environ['LLM_PROVIDER'] = 'surf'

from src.orchestrator.orchestrator import Orchestrator
from src.standards import METADATA_STANDARDS, get_schema_for_standard

MODEL = 'Qwen/Qwen3.6-35B-A3B-FP8'
TOPOLOGY = 'fast'
MAX_NEW_RUNS = int(os.getenv('M3_MAX_NEW_RUNS', '-1'))  # -1 runs all pending files
PANGAEA_STANDARD = 'pangaea_descriptive_technical_standard'
INAT_STANDARD = 'croissant_inaturalist_standard'
PANGAEA_FIELDS = ('name', 'description', 'keywords', 'spatialCoverage',
                  'temporalCoverage', 'spatial', 'temporal')
INAT_FIELDS = ('spatialCoverage', 'temporalCoverage')
DATA_DIRS = {
    'pangaea': ROOT / 'data/experiment/pangaea_datasets',
    'inaturalist': ROOT / 'data/experiment/inaturalist_100_species_1000_obs/inaturalist_100_species_1000_obs',
}
PRIOR_OUT = ROOT / 'outputs/m3_fast_20_default-text-large'
OUT = ROOT / 'outputs/m3_fast_all_Qwen3.6-35B-A3B-FP8'
OUT.mkdir(parents=True, exist_ok=True)

for corpus, data_dir in DATA_DIRS.items():
    if not data_dir.is_dir():
        raise FileNotFoundError(data_dir)
# if MODEL != 'default-text-large' or TOPOLOGY != 'fast':
#     raise ValueError('This notebook is pinned to the requested model and topology')
assert len(get_schema_for_standard(PANGAEA_STANDARD).model_fields) == 7
assert len(get_schema_for_standard(INAT_STANDARD).model_fields) == 2
print('Project:', ROOT)
print('Output:', OUT)
print('Prior successes read from:', PRIOR_OUT)
print('New M³ runs allowed this session:', 'all pending' if MAX_NEW_RUNS < 0 else MAX_NEW_RUNS)
print('SURF credentials present:', bool(os.getenv('SURF_API_KEY') and os.getenv('SURF_API_BASE')))


Project: /home/com3dian/Github/metadata_agent
Output: /home/com3dian/Github/metadata_agent/outputs/m3_fast_all_default-text-large
Prior successes read from: /home/com3dian/Github/metadata_agent/outputs/m3_fast_20_default-text-large
New M³ runs allowed this session: all pending
SURF credentials present: True


In [2]:
def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def write_json_atomic(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    temporary.replace(path)


def m3_result_path(root: Path, corpus: str, stem: str) -> Path:
    return root / 'm3' / corpus / f'{stem}.json'


def valid_m3_success(path: Path, item: dict, corpus: str) -> bool:
    if not path.is_file():
        return False
    try:
        record = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        return False
    standard = PANGAEA_STANDARD if corpus == 'pangaea' else INAT_STANDARD
    fields = PANGAEA_FIELDS if corpus == 'pangaea' else INAT_FIELDS
    return (record.get('status') == 'ok'
            and record.get('model') == MODEL
            and record.get('topology') == TOPOLOGY
            and record.get('standard') == standard
            and record.get('source_sha256') == item['source_sha256']
            and isinstance(record.get('metadata'), dict)
            and set(fields).issubset(record['metadata']))


manifest = {'model': MODEL, 'topology': TOPOLOGY, 'datasets': {}}
for corpus, data_dir in DATA_DIRS.items():
    selected = []
    for source in sorted(data_dir.glob('*.csv')):
        selected.append({'id': source.stem, 'source_sha256': sha256_file(source)})
    if not selected:
        raise FileNotFoundError(f'No CSV files in {data_dir}')
    manifest['datasets'][corpus] = selected
write_json_atomic(OUT / 'manifest.json', manifest)

adopted = 0
for corpus, selected in manifest['datasets'].items():
    for item in selected:
        destination = m3_result_path(OUT, corpus, item['id'])
        if destination.exists():
            continue
        prior = m3_result_path(PRIOR_OUT, corpus, item['id'])
        if valid_m3_success(prior, item, corpus):
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(prior, destination)
            adopted += 1

for corpus, selected in manifest['datasets'].items():
    ready = sum(valid_m3_success(m3_result_path(OUT, corpus, item['id']), item, corpus)
                for item in selected)
    print(corpus, len(selected), 'datasets;', ready, 'successes already saved')
print('Copied from the 20-dataset folder without rerunning:', adopted)
print('Manifest:', OUT / 'manifest.json')


pangaea 75 datasets; 0 successes already saved
inaturalist 100 datasets; 0 successes already saved
Copied from the 20-dataset folder without rerunning: 0
Manifest: /home/com3dian/Github/metadata_agent/outputs/m3_fast_all_default-text-large/manifest.json


In [3]:
import logging

log = logging.getLogger()
log.setLevel(logging.INFO)
for handler in log.handlers[:]:
    log.removeHandler(handler)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
log.addHandler(handler)


def run_m3_one(corpus: str, item: dict) -> dict:
    source = DATA_DIRS[corpus] / f"{item['id']}.csv"
    standard = PANGAEA_STANDARD if corpus == 'pangaea' else INAT_STANDARD
    fields = PANGAEA_FIELDS if corpus == 'pangaea' else INAT_FIELDS
    base = {'dataset_id': item['id'], 'csv': str(source), 'source_sha256': item['source_sha256'],
            'method': 'M3_fast', 'provider': 'surf', 'model': MODEL,
            'topology': TOPOLOGY, 'standard': standard,
            'finished_utc': datetime.now(timezone.utc).isoformat()}
    started = time.monotonic()
    orchestrator = None
    try:
        orchestrator = Orchestrator(topology_name=TOPOLOGY, model_name=MODEL, provider='surf')
        result = orchestrator.run(
            source=str(source), metadata_standard=METADATA_STANDARDS[standard],
            metadata_standard_name=standard, name=item['id'],
        )
        metadata = result.final_metadata if result is not None else None
        if result is None or not result.success:
            raise RuntimeError((result.error if result is not None else 'Planning failed') or 'Run failed')
        if not isinstance(metadata, dict) or not set(fields).issubset(metadata):
            raise ValueError(f'Missing requested metadata fields: {sorted(set(fields)-set(metadata or {}))}')
        metadata = get_schema_for_standard(standard).model_validate(metadata).model_dump(
            mode='json', by_alias=True
        )
        return {**base, 'status': 'ok', 'metadata': metadata,
                'plan_steps_count': result.plan_steps_count,
                'steps_completed': result.steps_completed,
                'trace_path': result.trace_path,
                'runtime_seconds': round(time.monotonic()-started, 2)}
    except Exception as exc:
        return {**base, 'status': 'failed', 'error': f'{type(exc).__name__}: {exc}',
                'trace_path': getattr(orchestrator, 'last_trace_path', None),
                'runtime_seconds': round(time.monotonic()-started, 2)}


if MAX_NEW_RUNS != 0 and not (os.getenv('SURF_API_KEY') and os.getenv('SURF_API_BASE')):
    raise RuntimeError('SURF_API_KEY and SURF_API_BASE are required for new M³ runs')

new_runs = 0
for corpus, selected in manifest['datasets'].items():
    for item in tqdm(selected, desc=f'M³ fast {corpus}'):
        path = m3_result_path(OUT, corpus, item['id'])
        if valid_m3_success(path, item, corpus):
            continue
        if 0 <= MAX_NEW_RUNS <= new_runs:
            continue
        print(f'Starting M³ {corpus} {item["id"]}', flush=True)
        record = run_m3_one(corpus, item)
        write_json_atomic(path, record)
        new_runs += 1
        print(corpus, item['id'], record['status'],
              record.get('error', ''), 'trace:', record.get('trace_path'))
print('New M³ attempts:', new_runs)
for corpus, selected in manifest['datasets'].items():
    print(corpus, 'saved successes:',
          sum(valid_m3_success(m3_result_path(OUT, corpus, item['id']), item, corpus)
              for item in selected), '/', len(selected))


M³ fast pangaea:   0%|          | 0/75 [00:00<?, ?it/s]

Starting M³ pangaea 897882
2026-10-04 01:17:37,476 - INFO - root - PlanExecutor initialized with topology: fast
2026-10-04 01:17:37,476 - INFO - root -   Players per step: 2
2026-10-04 01:17:37,477 - INFO - root -   Debate rounds: 1
2026-10-04 01:17:37,477 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-10-04 01:17:37,477 - INFO - root - Orchestrator initialized with topology: fast
2026-10-04 01:17:37,478 - INFO - root - Run trace: /home/com3dian/Github/metadata_agent/outputs/traces/897882_c34ef01473be45229a88faab86e5e425.jsonl
2026-10-04 01:17:37,478 - INFO - root - ============================================================
2026-10-04 01:17:37,478 - INFO - root - STARTING ORCHESTRATION
2026-10-04 01:17:37,479 - INFO - root - Context: 897882
2026-10-04 01:17:37,479 - INFO - root - Type: single_csv
2026-10-04 01:17:37,479 - INFO - root - Resources: ['

M³ fast inaturalist:   0%|          | 0/100 [00:00<?, ?it/s]

Starting M³ inaturalist acianthus_fornicatus
2026-10-04 04:04:03,372 - INFO - root - PlanExecutor initialized with topology: fast
2026-10-04 04:04:03,373 - INFO - root -   Players per step: 2
2026-10-04 04:04:03,373 - INFO - root -   Debate rounds: 1
2026-10-04 04:04:03,374 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-10-04 04:04:03,374 - INFO - root - Orchestrator initialized with topology: fast
2026-10-04 04:04:03,375 - INFO - root - Run trace: /home/com3dian/Github/metadata_agent/outputs/traces/acianthus_fornicatus_e37b06792e574c23968b0f1a52a6a5f2.jsonl
2026-10-04 04:04:03,375 - INFO - root - ============================================================
2026-10-04 04:04:03,375 - INFO - root - STARTING ORCHESTRATION
2026-10-04 04:04:03,376 - INFO - root - Context: acianthus_fornicatus
2026-10-04 04:04:03,376 - INFO - root - Type: single_csv
2026-10

/home/com3dian/Github/metadata_agent/src/tools/context_tools.py:441: FutureWarning: Parsed string "5H2KJ" included an un-recognized timezone "KJ". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  return pd.to_datetime(series, errors="coerce", utc=True, format="mixed")
/home/com3dian/Github/metadata_agent/src/tools/context_tools.py:441: FutureWarning: Parsed string "5H2KJ" included an un-recognized timezone "KJ". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  return pd.to_datetime(series, errors="coerce", utc=True, format="mixed")


2026-10-04 04:28:56,553 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-10-04 04:28:56,555 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-10-04 04:28:56,632 - INFO - root - Player 'spatial_temporal_specialist_2': LLM tool loop iteration 2 — model requested 4 tool(s): get_temporal_extent, get_temporal_extent, get_temporal_extent, get_spatial_extent
2026-10-04 04:28:56,642 - INFO - root - Player 'spatial_temporal_specialist_1': LLM tool loop iteration 2 — model requested 4 tool(s): get_temporal_extent, get_temporal_extent, get_temporal_extent, get_spatial_extent
2026-10-04 04:28:58,201 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-10-04 04:28:58,221 - INFO - root - Player 'spatial_temporal_specialist_1': LLM tool loop iteration 3 — model requested 2 tool(s): analyze_temporal_column, analyze_spatial_column
2026-10-

/home/com3dian/Github/metadata_agent/src/tools/context_tools.py:441: FutureWarning: Parsed string "5H9PJ" included an un-recognized timezone "PJ". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  return pd.to_datetime(series, errors="coerce", utc=True, format="mixed")
/home/com3dian/Github/metadata_agent/src/tools/context_tools.py:441: FutureWarning: Parsed string "5H9PH" included an un-recognized timezone "PH". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  return pd.to_datetime(series, errors="coerce", utc=True, format="mixed")
/home/com3dian/Github/metadata_agent/src/tools/context_tools.py:441: FutureWarning: Parsed string "5H9PJ" included an un-recognized timezone "PJ". Dropping unrecognized timezones

2026-10-04 04:32:08,854 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-10-04 04:32:08,854 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-10-04 04:32:08,917 - INFO - root - Player 'spatial_temporal_specialist_2': LLM tool loop iteration 2 — model requested 4 tool(s): get_temporal_extent, get_temporal_extent, get_temporal_extent, get_spatial_extent
2026-10-04 04:32:08,925 - INFO - root - Player 'spatial_temporal_specialist_1': LLM tool loop iteration 2 — model requested 4 tool(s): get_temporal_extent, get_temporal_extent, get_temporal_extent, get_spatial_extent
2026-10-04 04:32:10,084 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-10-04 04:32:10,099 - INFO - root - Player 'spatial_temporal_specialist_2': LLM tool loop iteration 3 — model requested 1 tool(s): analyze_temporal_column
2026-10-04 04:32:10,594 - INFO -